In [ ]:
#mount the drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import datetime
from statsmodels.tsa.stattools import coint
from itertools import combinations

In [ ]:
Data=pd.read_csv("/content/drive/My Drive/242B Project_LSTM+Sentiment+MAB/Train_Dataset_No_Sentiment.csv",header=None)

raw=Data.copy()
# Row 0 = tickers, row 1 = price fields
tickers = raw.iloc[0]
fields = raw.iloc[1]

# Actual data starts at row 3, skipping the 'Date' header row which was read as data
df = raw.iloc[3:].copy()

# First column is Date
df = df.rename(columns={0: "Date"})
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date")

# Build MultiIndex columns: (ticker, field)
new_cols = []
for ticker, field in zip(tickers[1:], fields[1:]):
    new_cols.append((ticker, field.lower()))

df.columns = pd.MultiIndex.from_tuples(new_cols, names=["ticker", "field"])

# Convert values to numeric
df = df.apply(pd.to_numeric)
df.head()

ticker            APA                                                   COP  \
field           close       high        low       open    volume      close   
Date                                                                          
2022-01-03  24.343285  24.412688  23.406337  23.493092   9319700  63.208145   
2022-01-04  25.557844  25.974267  24.811757  24.898512  13218200  65.949982   
2022-01-05  24.638248  26.104398  24.612223  26.026319   9212700  64.818977   
2022-01-06  25.696651  25.948238  24.967914  25.471090   7626800  67.252373   
2022-01-07  25.740030  26.286582  25.523144  25.896188   8355400  69.094551   

ticker                                                 ...        VLO  \
field            high        low       open    volume  ...      close   
Date                                                   ...              
2022-01-03  63.370944  61.708697  61.717268   5769900  ...  67.498604   
2022-01-04  66.301277  63.645115  63.885025   9189300  ...  68.697388   
2022-01-05  67.063860  64.707586  66.815386   9034500  ...  68.426109   
2022-01-06  67.509424  65.890020  66.678300   8679000  ...  70.141151   
2022-01-07  69.343032  67.158120  67.577970  10838800  ...  70.876160   

ticker                                                      XOM             \
field            high        low       open   volume      close       high   
Date                                                                         
2022-01-03  68.434869  66.124832  66.326088  3568200  54.760239  54.811946   
2022-01-04  69.467398  68.137379  68.399879  4171700  56.819996  57.044072   
2022-01-05  69.861132  68.111104  69.344877  3999800  57.526684  58.267852   
2022-01-06  70.412404  69.091130  70.001146  4248200  58.879753  59.017648   
2022-01-07  71.252416  69.957392  70.508655  3723200  59.362370  59.620919   

ticker                                      
field             low       open    volume  
Date                                        
2022-01-03  52.752189  52.778046  24282400  
2022-01-04  55.242858  55.268712  38584000  
2022-01-05  57.293994  57.311228  34033300  
2022-01-06  57.802474  58.603970  30668500  
2022-01-07  58.586734  59.052113  23985400  

[5 rows x 70 columns]

In [ ]:


def compute_metrics(strategy_returns, periods_per_year=252):
    r = strategy_returns.dropna()

    if len(r) == 0:
        return {
            "Cumulative Return": np.nan,
            "Annualized Volatility": np.nan,
            "Sharpe Ratio": np.nan,
            "Max Drawdown": np.nan,
            "Win Rate": np.nan,
        }

    cumulative_curve = (1 + r).cumprod()

    cumulative_return = cumulative_curve.iloc[-1] - 1

    annualized_volatility = r.std() * np.sqrt(periods_per_year)

    if r.std() == 0:
        sharpe = np.nan
    else:
        sharpe = (r.mean() / r.std()) * np.sqrt(periods_per_year)

    peak = cumulative_curve.cummax()
    drawdown = (cumulative_curve - peak) / peak
    max_drawdown = drawdown.min()

    win_rate = (r > 0).mean()

    return {
        "Cumulative Return": cumulative_return,
        "Annualized Volatility": annualized_volatility,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown,
        "Win Rate": win_rate,
    }

In [ ]:
tickers = [
    "XOM",
    "CVX",
    "COP",
    "OXY",
   # "PXD":  "Pioneer Natural Resources",
    "SLB",
    "HAL",
    "EOG",
    "DVN",
    "MPC",

    # stocks added by Ryan

    #"PLC": "Principal U.S. Large-Cap Multi-Factor ETF",
    "FANG",
    "PSX",
    "VLO",
    "CTRA",
    "APA"
]

## Single Stock Evaluation

### Momentum (20 day lookback)

In [ ]:

def time_series_momentum_strategy(Data_Frame, stock_name,start_date, end_date,price_col="close", lookback=20, threshold=0.0, allow_short=True):
    # Filter for the specific stock
    stock_df = Data_Frame[stock_name].copy()

    #start from the first available date
    # on or after start_date, handling non-trading days.
    out = stock_df.loc[start_date:].copy()

    out["return"] = out[price_col].pct_change() # return for each day (rate of change)
    out["momentum"] = out[price_col].pct_change(lookback) # return for 20 days lookback

    if allow_short:
        out["signal"] = np.select(
            [out["momentum"] > threshold, out["momentum"] < -threshold],
            [1, -1],
            default=0
        )
    else:
        out["signal"] = np.where(out["momentum"] > threshold, 1, 0)

    out["position"] = out["signal"].shift(1).fillna(0)

    out["trade"] = out["position"].diff().abs().fillna(0)
    out["strategy_return"] = out["position"] * out["return"]

    out["cumulative_strategy_return"] = (1 + out["strategy_return"].fillna(0)).cumprod()
    out["cumulative_asset_return"] = (1 + out["return"].fillna(0)).cumprod()

    # Slice after features are computed
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    return out

In [ ]:
result_mm = time_series_momentum_strategy(
    df,
    "APA",
    start_date="2022-01-03",
    end_date="2023-01-03",
    price_col="close",
    lookback=20,
    threshold=0.02,
    allow_short=True
)

print(result_mm['position'].loc["2026-02-01":"2026-04-10"])

Series([], Name: position, dtype: float64)


In [ ]:
metrics = compute_metrics(result_mm["strategy_return"])

for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: -0.6122
Annualized Volatility: 0.5406
Sharpe Ratio: -1.4832
Max Drawdown: -0.6367
Win Rate: 0.3745


## Mean Reversion

In [ ]:
import numpy as np
import pandas as pd

def mean_reversion_strategy(
    Data_Frame,
    stock_name,
    price_col="Close",
    lookback=20,
    entry_z=2.0,
    exit_z=0.5,
    allow_short=True,
    start_date=None,
    end_date=None
):
    stock_df = Data_Frame[stock_name].copy()

    #start from the first available date
    # on or after start_date, handling non-trading days.
    out = stock_df.loc[start_date:].copy()

    # 1-period asset return
    out["return"] = out[price_col].pct_change()

    # Rolling mean and std = estimate of "normal" price behavior
    out["rolling_mean"] = out[price_col].rolling(window=lookback).mean()
    out["rolling_std"] = out[price_col].rolling(window=lookback).std()

    # Z-score: how far price is from its recent average
    out["z_score"] = (out[price_col] - out["rolling_mean"]) / out["rolling_std"]

    # Start flat
    out["signal"] = 0

    if allow_short:
        # Price too low relative to average → long
        out.loc[out["z_score"] < -entry_z, "signal"] = 1

        # Price too high relative to average → short
        out.loc[out["z_score"] > entry_z, "signal"] = -1
    else:
        # Long-only version
        out.loc[out["z_score"] < -entry_z, "signal"] = 1

    # Exit when price comes back close to average
    out.loc[out["z_score"].abs() < exit_z, "signal"] = 0

    # Carry position forward until new signal or exit
    out["position"] = out["signal"].replace(0, np.nan).ffill().fillna(0)

    # If exit condition is met, force position to 0
    out.loc[out["z_score"].abs() < exit_z, "position"] = 0

    # Shift position to avoid lookahead bias
    out["position"] = out["position"].shift(1).fillna(0)

    # Trade size/change
    out["trade"] = out["position"].diff().abs().fillna(0)

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative performance
    out["cumulative_strategy_return"] = (1 + out["strategy_return"].fillna(0)).cumprod()
    out["cumulative_asset_return"] = (1 + out["return"].fillna(0)).cumprod()

    # Slice after features are computed
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    return out

In [ ]:
result_mr = mean_reversion_strategy(
    df,
    "APA",
    price_col="close",
    lookback=20,
    entry_z=2.0,
    exit_z=0.5,
    allow_short=True,
    start_date="2022-01-03",
    end_date="2023-01-03"
)

result_mr[[
    "close",
    "rolling_mean",
    "z_score",
    "signal",
    "position",
    "strategy_return"
]].tail()

field,close,rolling_mean,z_score,signal,position,strategy_return
Date,,,,,,
2022-12-27,42.006996,39.810534,1.510230,0,1.0,0.008254
2022-12-28,39.837872,39.748370,0.062876,0,1.0,-0.051637
2022-12-29,40.481552,39.706928,0.558410,0,0.0,0.000000
2022-12-30,41.160515,39.705164,1.050624,0,1.0,0.016772
2023-01-03,38.462322,39.551297,-0.812103,0,1.0,-0.065553


In [ ]:
metrics_mr = compute_metrics(result_mr["strategy_return"])

for k, v in metrics_mr.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: 0.4947
Annualized Volatility: 0.4802
Sharpe Ratio: 1.0818
Max Drawdown: -0.3654
Win Rate: 0.3625


### Closing Range Breakout

In [ ]:

def closing_range_breakout(
    Data_Frame,
    stock_name,
    price_col="close",
    lookback=20,
    upper_th=0.8,
    lower_th=0.2,
    allow_short=True,
    start_date=None,
    end_date=None
):
    stock_df = Data_Frame[stock_name].copy()

    #start from the first available date
    # on or after start_date, handling non-trading days.
    out = stock_df.loc[start_date:].copy()

    # Returns
    out["return"] = out[price_col].pct_change()

    # Rolling high and low
    out["rolling_high"] = out[price_col].rolling(lookback).max()
    out["rolling_low"]  = out[price_col].rolling(lookback).min()

    # Closing range
    out["cr"] = (out[price_col] - out["rolling_low"]) / (
        out["rolling_high"] - out["rolling_low"]
    )

    # Signal
    if allow_short:
        out["signal"] = np.select(
            [out["cr"] > upper_th, out["cr"] < lower_th],
            [1, -1],
            default=0
        )
    else:
        out["signal"] = np.where(out["cr"] > upper_th, 1, 0)

    # Position (avoid lookahead bias)
    out["position"] = out["signal"].shift(1).fillna(0)

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative returns
    out["cumulative_strategy_return"] = (1 + out["strategy_return"].fillna(0)).cumprod()
    out["cumulative_asset_return"] = (1 + out["return"].fillna(0)).cumprod()

    # Slice after computing features
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    return out

In [ ]:
result_crb = closing_range_breakout(
    df,
    "APA",
    price_col="close",
    lookback=20,
    upper_th=0.8,
    lower_th=0.2,
    allow_short=True,
    start_date="2022-01-03",
    end_date="2023-01-03"
)

result_crb[[
    "close",
    "rolling_high",
    "rolling_low",
    "cr",
    "signal",
    "position",
    "strategy_return",
    "cumulative_strategy_return"
]].head(40)


field,close,rolling_high,rolling_low,cr,signal,position,strategy_return,cumulative_strategy_return
Date,,,,,,,,
2022-01-03,24.343285,NaN,NaN,NaN,0,0.0,NaN,1.000000
2022-01-04,25.557844,NaN,NaN,NaN,0,0.0,0.000000,1.000000
2022-01-05,24.638248,NaN,NaN,NaN,0,0.0,-0.000000,1.000000
2022-01-06,25.696651,NaN,NaN,NaN,0,0.0,0.000000,1.000000
2022-01-07,25.740030,NaN,NaN,NaN,0,0.0,0.000000,1.000000
2022-01-10,25.523146,NaN,NaN,NaN,0,0.0,-0.000000,1.000000
2022-01-11,27.761408,NaN,NaN,NaN,0,0.0,0.000000,1.000000
2022-01-12,28.325312,NaN,NaN,NaN,0,0.0,0.000000,1.000000
2022-01-13,27.839487,NaN,NaN,NaN,0,0.0,-0.000000,1.000000


In [ ]:
metrics_crb = compute_metrics(result_crb["strategy_return"])

for k, v in metrics_crb.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: -0.4144
Annualized Volatility: 0.4332
Sharpe Ratio: -1.0202
Max Drawdown: -0.4407
Win Rate: 0.2669


### Turnaround Tueasday - IBS (Internal Bar Strength)

In [ ]:
def turnaround_tuesday_ibs(
    Data_Frame,
    stock_name,
    ibs_threshold=0.2,
    start_date=None,
    end_date=None
):
    stock_df = Data_Frame[stock_name].copy()

    #start from the first available date
    # on or after start_date, handling non-trading days.
    out = stock_df.loc[start_date:].copy()

    # Returns
    out["return"] = out["close"].pct_change()

    # IBS
    out["ibs"] = (out["close"] - out["low"]) / (out["high"] - out["low"])

    # Day of week (Monday = 0, Tuesday = 1, ...)
    out["weekday"] = pd.to_datetime(out.index).weekday

    # Initialize position
    out["position"] = 0

    # Entry: Monday IBS signal → position on Tuesday
    entry_signal = (out["weekday"] == 0) & (out["ibs"] < ibs_threshold)

    # Shift to enter on Tuesday
    out.loc[entry_signal.shift(1, fill_value=False), "position"] = 1

    # Exit after 1 day → automatically handled since no carry forward

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative returns
    out["cumulative_strategy_return"] = (1 + out["strategy_return"].fillna(0)).cumprod()
    out["cumulative_asset_return"] = (1 + out["return"].fillna(0)).cumprod()

    # Slice
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    return out

In [ ]:
result_ibs = turnaround_tuesday_ibs(
    df,
    "APA",
    ibs_threshold=0.2,
    start_date="2022-01-03",
    end_date="2023-01-03"
)

result_ibs[result_ibs["position"]!=0]

field,close,high,low,open,volume,return,ibs,weekday,position,strategy_return,cumulative_strategy_return,cumulative_asset_return
Date,,,,,,,,,,,,
2022-02-15,28.069305,28.199943,26.911000,27.041636,6883700,0.002800,0.898648,1,1,0.002800,1.002800,1.153062
2022-03-08,33.643108,35.881337,32.624146,34.078561,16485200,0.018724,0.312835,1,1,0.018724,1.021577,1.382028
2022-04-12,36.734810,37.675386,36.534502,36.560628,5817500,0.027027,0.175572,1,1,0.027027,1.049187,1.509033
2022-05-10,32.646526,34.480599,31.546080,33.371421,11971800,-0.003997,0.375001,1,1,-0.003997,1.044993,1.341090
2022-09-27,28.643990,29.283833,28.030440,28.775466,10516700,0.026382,0.489511,1,1,0.026382,1.072562,1.176669
2022-10-11,35.480682,36.409773,34.937254,35.147612,7325300,-0.016760,0.369047,1,1,-0.016760,1.054586,1.457514
2022-10-18,34.902199,35.892644,34.157176,35.366742,7240600,-0.006982,0.429292,1,1,-0.006982,1.047223,1.433751
2022-11-15,43.241467,43.400183,41.028249,41.839469,5814400,0.038763,0.933085,1,1,0.038763,1.087816,1.776320
2022-12-06,38.682758,40.737258,38.427051,39.732053,5685200,-0.040254,0.110686,1,1,-0.040254,1.044028,1.589053


In [ ]:
metrics_ibs = compute_metrics(result_ibs["strategy_return"])

for k, v in metrics_ibs.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: 0.0440
Annualized Volatility: 0.0727
Sharpe Ratio: 0.6311
Max Drawdown: -0.0403
Win Rate: 0.0199


### ATR (Average True Value) Breakout

In [ ]:

def atr_breakout(
    Data_Frame,
    stock_name,
    high_col="high",
    low_col="low",
    close_col="close",
    atr_lookback=14,
    multiplier=2.0,
    allow_short=True,
    start_date=None,
    end_date=None
):
    stock_df = Data_Frame[stock_name].copy()

    #start from the first available date
    # on or after start_date, handling non-trading days.
    out = stock_df.loc[start_date:].copy()

    # Returns
    out["return"] = out[close_col].pct_change()

    # True Range
    out["tr"] = np.maximum.reduce([
        out[high_col] - out[low_col],
        (out[high_col] - out[close_col].shift(1)).abs(),
        (out[low_col] - out[close_col].shift(1)).abs()
    ])

    # ATR
    out["atr"] = out["tr"].rolling(atr_lookback).mean()

    # Breakout levels (use previous values to avoid lookahead bias)
    out["upper"] = out[close_col].shift(1) + multiplier * out["atr"].shift(1)
    out["lower"] = out[close_col].shift(1) - multiplier * out["atr"].shift(1)

    # Signal
    if allow_short:
        out["signal"] = np.select(
            [out[close_col] > out["upper"], out[close_col] < out["lower"]],
            [1, -1],
            default=0
        )
    else:
        out["signal"] = np.where(out[close_col] > out["upper"], 1, 0)

    # Position
    out["position"] = out["signal"].shift(1).fillna(0)

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative returns
    out["cumulative_strategy_return"] = (1 + out["strategy_return"].fillna(0)).cumprod()
    out["cumulative_asset_return"] = (1 + out["return"].fillna(0)).cumprod()

    # Slice
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    return out

In [ ]:
result_atr = atr_breakout(
    df,
    "APA",
    high_col="high",
    low_col="low",
    close_col="close",
    atr_lookback=14,
    multiplier=2.0,
    allow_short=True,
    start_date="2022-01-03",
    end_date="2023-01-03"
)


result_atr[["close","tr","atr","upper","lower","signal","position","strategy_return","cumulative_strategy_return", "cumulative_asset_return"]].tail()


field,close,tr,atr,upper,lower,signal,position,strategy_return,cumulative_strategy_return,cumulative_asset_return
Date,,,,,,,,,,
2022-12-27,42.006996,0.784771,1.795639,45.472302,37.853908,0,0.0,0.0,1.035458,1.725609
2022-12-28,39.837872,2.292570,1.867439,45.598274,38.415719,0,0.0,-0.0,1.035458,1.636504
2022-12-29,40.481552,1.640072,1.816423,43.572749,36.102994,0,0.0,0.0,1.035458,1.662945
2022-12-30,41.160515,1.119834,1.783042,44.114398,36.848707,0,0.0,0.0,1.035458,1.690837
2023-01-03,38.462322,3.227247,1.856102,44.726598,37.594431,0,0.0,-0.0,1.035458,1.579997


In [ ]:
metrics_atr = compute_metrics(result_atr["strategy_return"])

for k, v in metrics_atr.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: 0.0355
Annualized Volatility: 0.0316
Sharpe Ratio: 1.1213
Max Drawdown: 0.0000
Win Rate: 0.0080


### Pairs Trading

In [ ]:

def pairs_trading(
    Data_Frame,
    tickers,
    close_col="close",
    lookback=60,
    z_entry=2.0,
    z_exit=0.5,
    start_date=None,
    end_date=None
):
    df = Data_Frame.copy()

    # Select only the 'close' price for the specified tickers
    # Use .loc with a MultiIndex slice to get the correct columns
    prices = df.loc[:, (tickers, close_col)].copy()

    # Rename columns to just the ticker names for easier access later
    # The columns are now (ticker, close_col), we want just ticker
    prices.columns = prices.columns.get_level_values(0)

    # Slice by date
    if start_date is not None:
        prices = prices.loc[start_date:]
    if end_date is not None:
        prices = prices.loc[:end_date]

    # Drop tickers with too many missing values
    prices = prices.dropna(axis=1)

    # Need at least 2 stocks
    if prices.shape[1] < 2:
        raise ValueError("Need at least two valid tickers with price data.")

    # --------------------------------------------------
    # 1. Choose best pair using cointegration test
    # --------------------------------------------------

    best_pair = None
    best_pvalue = np.inf

    for stock_a, stock_b in combinations(prices.columns, 2):
        series_a = prices[stock_a].dropna()
        series_b = prices[stock_b].dropna()

        common_index = series_a.index.intersection(series_b.index)
        series_a = series_a.loc[common_index]
        series_b = series_b.loc[common_index]

        if len(series_a) < lookback:
            continue

        score, pvalue, _ = coint(series_a, series_b)

        if pvalue < best_pvalue:
            best_pvalue = pvalue
            best_pair = (stock_a, stock_b)

    if best_pair is None:
        raise ValueError("No valid pair found.")

    stock_a, stock_b = best_pair

    # --------------------------------------------------
    # 2. Run pairs trading on selected pair
    # --------------------------------------------------

    out = prices[[stock_a, stock_b]].dropna().copy()

    out["log_A"] = np.log(out[stock_a])
    out["log_B"] = np.log(out[stock_b])

    # Estimate hedge ratio: log_A = alpha + beta * log_B
    X = sm.add_constant(out["log_B"])
    model = sm.OLS(out["log_A"], X).fit()
    beta = model.params["log_B"]

    # Spread
    out["spread"] = out["log_A"] - beta * out["log_B"]

    # Rolling mean/std of spread
    out["spread_mean"] = out["spread"].rolling(lookback).mean()
    out["spread_std"] = out["spread"].rolling(lookback).std()

    # Z-score
    out["z_score"] = (out["spread"] - out["spread_mean"]) / out["spread_std"]

    # Signal:
    # +1 = long spread = long A, short B
    # -1 = short spread = short A, long B
    out["signal"] = 0
    out.loc[out["z_score"] < -z_entry, "signal"] = 1
    out.loc[out["z_score"] > z_entry, "signal"] = -1

    # Exit when spread comes back near mean
    out.loc[out["z_score"].abs() < z_exit, "signal"] = 0

    # Stateful position
    out["position"] = out["signal"].replace(0, np.nan).ffill().fillna(0)

    # Force exit when z-score is close to 0
    out.loc[out["z_score"].abs() < z_exit, "position"] = 0

    # Shift to avoid lookahead bias
    out["position"] = out["position"].shift(1).fillna(0)

    # Returns
    out["ret_A"] = out[stock_a].pct_change()
    out["ret_B"] = out[stock_b].pct_change()

    # Strategy return
    out["strategy_return"] = out["position"] * (out["ret_A"] - beta * out["ret_B"])

    # Cumulative return
    out["cumulative_strategy_return"] = (
        1 + out["strategy_return"].fillna(0)
    ).cumprod()

    # Useful metadata columns
    out["stock_A"] = stock_a
    out["stock_B"] = stock_b
    out["beta"] = beta
    out["cointegration_pvalue"] = best_pvalue

    return out

In [ ]:
result_pt = pairs_trading(
    df,
    tickers,
    close_col="close",
    lookback=20,
    z_entry=2.0,
    z_exit=0.5,
    start_date="2022-01-03",
    end_date="2023-01-03"
)

# Dynamically get the chosen stock names from the result_pt DataFrame
stock_a = result_pt['stock_A'].iloc[0]
stock_b = result_pt['stock_B'].iloc[0]

# Display relevant columns generated by the pairs_trading function, using dynamic stock names
result_pt[[stock_a, stock_b, "spread", "z_score", "signal", "position", "strategy_return", "cumulative_strategy_return"]].tail()

ticker,XOM,MPC,spread,z_score,signal,position,strategy_return,cumulative_strategy_return
Date,,,,,,,,
2022-12-27,98.731544,108.886047,0.751010,0.342105,0,0.0,0.00000,1.237231
2022-12-28,97.109734,106.742249,0.750733,0.217262,0,0.0,-0.00000,1.237231
2022-12-29,97.844467,108.289520,0.746484,-0.455364,0,0.0,-0.00000,1.237231
2022-12-30,98.830093,108.485268,0.755028,0.674040,0,0.0,0.00000,1.237231
2023-01-03,95.434196,103.386749,0.759488,1.332928,0,1.0,0.00413,1.242341


In [ ]:
metrics_pt = compute_metrics(result_pt["strategy_return"])

for k, v in metrics_pt.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: 0.2423
Annualized Volatility: 0.1913
Sharpe Ratio: 1.2347
Max Drawdown: -0.1301
Win Rate: 0.3267


### Volatility Expension

In [ ]:
def volatility_expansion(
    Data_Frame,
    stock_name,
    close_col="close",
    vol_lookback=20,
    vol_multiplier=1.5,
    allow_short=True,
    start_date=None,
    end_date=None
):
    out = Data_Frame[stock_name].copy()

    # Slice
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    # Returns
    out["return"] = out[close_col].pct_change()

    # Volatility
    out["vol"] = out["return"].rolling(vol_lookback).std()

    # Previous volatility (baseline)
    out["vol_prev"] = out["vol"].rolling(vol_lookback).mean()

    # Signal
    out["signal"] = 0

    # Volatility expansion condition
    vol_expansion = out["vol"] > vol_multiplier * out["vol_prev"]

    if allow_short:
        out.loc[vol_expansion & (out["return"] > 0), "signal"] = 1
        out.loc[vol_expansion & (out["return"] < 0), "signal"] = -1
    else:
        out.loc[vol_expansion & (out["return"] > 0), "signal"] = 1

    # Position
    out["position"] = out["signal"].shift(1).fillna(0)

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative
    out["cumulative_strategy_return"] = (
        1 + out["strategy_return"].fillna(0)
    ).cumprod()

    out["cumulative_asset_return"] = (
        1 + out["return"].fillna(0)
    ).cumprod()

    return out

In [ ]:
result_ve = volatility_expansion(
    df,
    "APA",
    close_col="close",
    vol_lookback=20,
    vol_multiplier=1.5,
    allow_short=True,
    start_date="2022-01-03",
    end_date="2023-01-03"
    )

result_ve[["close","vol","vol_prev","signal","position","strategy_return","cumulative_strategy_return", "cumulative_asset_return"]].tail()

field,close,vol,vol_prev,signal,position,strategy_return,cumulative_strategy_return,cumulative_asset_return
Date,,,,,,,,
2022-12-27,42.006996,0.031195,0.030650,0,0.0,0.0,1.002157,1.725609
2022-12-28,39.837872,0.032481,0.030456,0,0.0,-0.0,1.002157,1.636504
2022-12-29,40.481552,0.032680,0.030272,0,0.0,0.0,1.002157,1.662945
2022-12-30,41.160515,0.032900,0.030147,0,0.0,0.0,1.002157,1.690837
2023-01-03,38.462322,0.035975,0.030350,0,0.0,-0.0,1.002157,1.579997


In [ ]:
metrics_ve = compute_metrics(result_ve["strategy_return"])

for k, v in metrics_ve.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: 0.0022
Annualized Volatility: 0.0403
Sharpe Ratio: 0.0737
Max Drawdown: -0.0315
Win Rate: 0.0080


### Moving Average Crossover

In [ ]:
def moving_average_crossover(
    Data_Frame,
    stock_name,
    close_col="close",
    short_window=20,
    long_window=50,
    allow_short=True,
    start_date=None,
    end_date=None
):
    out = Data_Frame[stock_name].copy()

    # Slice
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    # Returns
    out["return"] = out[close_col].pct_change()

    # Moving averages
    out["ma_short"] = out[close_col].rolling(short_window).mean()
    out["ma_long"] = out[close_col].rolling(long_window).mean()

    # Signal
    if allow_short:
        out["signal"] = np.where(
            out["ma_short"] > out["ma_long"], 1, -1
        )
    else:
        out["signal"] = np.where(
            out["ma_short"] > out["ma_long"], 1, 0
        )

    # Position
    out["position"] = out["signal"].shift(1).fillna(0)

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative
    out["cumulative_strategy_return"] = (
        1 + out["strategy_return"].fillna(0)
    ).cumprod()

    out["cumulative_asset_return"] = (
        1 + out["return"].fillna(0)
    ).cumprod()

    return out

In [ ]:
result_mac = moving_average_crossover(
    df,
    "APA",
    close_col="close",
    short_window=20,
    long_window=50,
    allow_short=True,
    start_date="2022-01-03",
    end_date="2023-01-03"
)

result_mac[["close","ma_short","ma_long","signal","position","strategy_return","cumulative_strategy_return", "cumulative_asset_return"]].tail()

field,close,ma_short,ma_long,signal,position,strategy_return,cumulative_strategy_return,cumulative_asset_return
Date,,,,,,,,
2022-12-27,42.006996,39.810534,40.116387,-1,-1.0,-0.008254,0.483375,1.725609
2022-12-28,39.837872,39.748370,40.210192,-1,-1.0,0.051637,0.508335,1.636504
2022-12-29,40.481552,39.706928,40.321779,-1,-1.0,-0.016158,0.500122,1.662945
2022-12-30,41.160515,39.705164,40.410308,-1,-1.0,-0.016772,0.491734,1.690837
2023-01-03,38.462322,39.551297,40.447518,-1,-1.0,0.065553,0.523968,1.579997


In [ ]:
metrics_mac = compute_metrics(result_mac["strategy_return"])

for k, v in metrics_mac.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: -0.4760
Annualized Volatility: 0.5921
Sharpe Ratio: -0.7963
Max Drawdown: -0.6583
Win Rate: 0.4821


### Single Srock Evaluation - Comparing all trading strategies

In [ ]:
strategies = {
    "Momentum": result_mm["strategy_return"],
    "Mean Reversion": result_mr["strategy_return"],
    "Closing Range Breakout": result_crb["strategy_return"],
    "Turnaround Tuesday IBS": result_ibs["strategy_return"],
    "ATR Breakout": result_atr["strategy_return"],
    "Pairs Trading": result_pt["strategy_return"],
    "Volatility Expansion": result_ve["strategy_return"],
    "Moving Average Crossover": result_mac["strategy_return"],

}

metrics_table = pd.DataFrame({
    name: compute_metrics(returns)
    for name, returns in strategies.items()
}).T

metrics_table

metrics_table.style.format({
    "Cumulative Return": "{:.2%}",
    "Annualized Volatility": "{:.2%}",
    "Sharpe Ratio": "{:.2f}",
    "Max Drawdown": "{:.2%}",
    "Win Rate": "{:.2%}",
})

,Cumulative Return,Annualized Volatility,Sharpe Ratio,Max Drawdown,Win Rate
Momentum,-61.22%,54.06%,-1.48,-63.67%,37.45%
Mean Reversion,49.47%,48.02%,1.08,-36.54%,36.25%
Closing Range Breakout,-41.44%,43.32%,-1.02,-44.07%,26.69%
Turnaround Tuesday IBS,4.40%,7.27%,0.63,-4.03%,1.99%
ATR Breakout,3.55%,3.16%,1.12,0.00%,0.80%
Pairs Trading,24.23%,19.13%,1.23,-13.01%,32.67%
Volatility Expansion,0.22%,4.03%,0.07,-3.15%,0.80%
Moving Average Crossover,-47.60%,59.21%,-0.80,-65.83%,48.21%


### Portfolio Construction

### EXP3 Bandit Algo